In [1]:
# Import SparkSession class from pyspark.sql module
# SparkSession is the entry point to use Spark functionality in PySpark
# Without SparkSession, we cannot create DataFrames or run Spark jobs

from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("BigData_Regression_Tutorial").getOrCreate()
# Create or get an existing Spark session
# SparkSession.builder is used to configure the Spark application

#spark = SparkSession.builder \

    # Set the name of the Spark application
    # This name appears in Spark UI and logs
    #.appName("BigData_Regression_Tutorial") \

    # getOrCreate() does two things:
    # 1️⃣ If a Spark session already exists → it returns that session
    # 2️⃣ If no session exists → it creates a new Spark session

   # .getOrCreate()

In [3]:
df = spark.read.csv("/content/drive/MyDrive/Big Data Foundation/housing_bigdata.csv",
                    header=True,          # First row contains column names
                    inferSchema=True)     # Automatically detect data types (int, double, etc.)

df.printSchema()   # Displays column names, data types, and nullability information

df.show()         # Shows first 20 rows of the DataFrame (triggers execution)


root
 |-- area: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- floors: integer (nullable = true)
 |-- parking: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- location_score: double (nullable = true)
 |-- school_rating: double (nullable = true)
 |-- hospital_distance: double (nullable = true)
 |-- crime_rate: double (nullable = true)
 |-- population_density: integer (nullable = true)
 |-- median_income: integer (nullable = true)
 |-- highway_distance: double (nullable = true)
 |-- price: double (nullable = true)

+----+--------+---------+------+-------+---+--------------+-------------+-----------------+----------+------------------+-------------+----------------+----------+
|area|bedrooms|bathrooms|floors|parking|age|location_score|school_rating|hospital_distance|crime_rate|population_density|median_income|highway_distance|     price|
+----+--------+---------+------+-------+---+--------------+--------

In [4]:
from pyspark.ml.feature import VectorAssembler   # Import VectorAssembler to combine feature columns into a single vector column

feature_columns = [                              # List of independent (input) feature column names
    'area','bedrooms','bathrooms','floors','parking',
    'age','location_score','school_rating',
    'hospital_distance','crime_rate',
    'population_density','median_income','highway_distance'
]

assembler = VectorAssembler(                     # Create VectorAssembler object
    inputCols=feature_columns,                   # Columns to combine into feature vector
    outputCol="features"                         # Name of the new output column containing combined features
)

data = assembler.transform(df)                   # Transform original DataFrame by adding the "features" column

data.select("features","price").show(10)          # Show first 5 rows of features vector and target variable (price)


+--------------------+----------+
|            features|     price|
+--------------------+----------+
|[1360.0,3.0,3.0,1...| 758351.48|
|[4272.0,5.0,2.0,3...|1030963.01|
|[3592.0,2.0,3.0,3...|1076900.82|
|[966.0,6.0,2.0,1....| 832084.23|
|[4926.0,3.0,3.0,2...|1332342.64|
|[3944.0,3.0,2.0,2...|1086245.37|
|[3671.0,6.0,1.0,2...| 968147.36|
|[3419.0,2.0,3.0,2...|1077835.41|
|[630.0,2.0,3.0,3....| 649542.61|
|[2185.0,4.0,1.0,1...| 823551.89|
+--------------------+----------+
only showing top 10 rows


In [5]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)   # Split dataset into 80% training data and 20% testing data (seed ensures same split every time)


In [6]:
from pyspark.ml.regression import LinearRegression   # Import Linear Regression algorithm from PySpark ML library

lr = LinearRegression(                               # Create Linear Regression model object
    featuresCol='features',                          # Column containing input feature vectors (X)
    labelCol='price'                                 # Column containing target variable (Y)
)

model = lr.fit(train_data)                           # Train the model using training dataset


In [7]:
training_summary = model.summary                     # Get training performance summary of the model

print("R2 Score:", training_summary.r2)              # Print R² score (how well model explains variance)

print("RMSE:", training_summary.rootMeanSquaredError)  # Print Root Mean Squared Error (measures prediction error magnitude)

print("MAE:", training_summary.meanAbsoluteError)    # Print Mean Absolute Error (average absolute difference between actual and predicted values)


R2 Score: 0.9867180246351412
RMSE: 25009.13070168982
MAE: 19954.00793841032


In [8]:
predictions = model.transform(test_data)   # Apply trained model on test data to generate predictions

predictions.select("features","price","prediction").show(5)   # Display first 5 rows with features, actual price, and predicted price


+--------------------+---------+------------------+
|            features|    price|        prediction|
+--------------------+---------+------------------+
|[500.0,1.0,1.0,1....|595773.94|  612433.244820607|
|[500.0,1.0,1.0,2....|624128.65| 570258.3521092302|
|[500.0,1.0,1.0,2....|495630.78| 489446.0988496768|
|[500.0,1.0,1.0,2....|636405.86| 594606.6373894623|
|[500.0,1.0,1.0,3....|420972.78|467580.28207685583|
+--------------------+---------+------------------+
only showing top 5 rows


In [9]:
from pyspark.ml.evaluation import RegressionEvaluator

# Create a RegressionEvaluator instance
evaluator_r2 = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="r2"
)
evaluator_rmse = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)
evaluator_mae = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="mae"
)

# Evaluate the model on the test data
r2 = evaluator_r2.evaluate(predictions)
rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print(f"R2 Score (Test): {r2}")
print(f"RMSE (Test): {rmse}")
print(f"MAE (Test): {mae}")

R2 Score (Test): 0.9867677526899693
RMSE (Test): 24984.032027079367
MAE (Test): 19937.287985897077


R2 Score: 0.9867180246351412
RMSE: 25009.13070168982
MAE: 19954.00793841032